<a href="https://colab.research.google.com/github/devanshh019/codeexpo/blob/main/competition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [75]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [76]:
df = pd.read_csv("/content/emergency_call_dataset_10000_realistic.csv");

In [77]:
df.head()

,call_id,call_text,emergency_type,emotion,sound_event,urgency_level
0,1,oh my god a man is yelling and hitting someone...,domestic_violence,fear,scream,high
1,2,listen flood water is entering houses and peop...,natural_disaster,panic,thunder,critical
2,3,can you help a bike hit a pedestrian and the r...,traffic_accident,stress,car_horn,high
3,4,can you help a man just snatched a bag and ran...,robbery,calm,scream,high
4,5,can you help a child fell into the swimming po...,drowning,fear,water_splash,critical


In [78]:
df.isnull().sum()

,0
call_id,0
call_text,0
emergency_type,0
emotion,0
sound_event,0
urgency_level,0


In [79]:
df['call_text'].duplicated().sum()

df.drop_duplicates(subset=['call_text'], inplace=True)

In [80]:
df.shape

(780, 6)

In [81]:
df['emergency_type'].unique()

array(['domestic_violence', 'natural_disaster', 'traffic_accident',
       'robbery', 'drowning', 'medical_emergency', 'suspicious_activity',
       'shooting', 'fire', 'non_emergency', 'construction_accident',
       'explosion', 'animal_attack'], dtype=object)

In [82]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['emergency_type'] = le.fit_transform(df['emergency_type'])

In [83]:
df.head()

,call_id,call_text,emergency_type,emotion,sound_event,urgency_level
0,1,oh my god a man is yelling and hitting someone...,2,fear,scream,high
1,2,listen flood water is entering houses and peop...,7,panic,thunder,critical
2,3,can you help a bike hit a pedestrian and the r...,12,stress,car_horn,high
3,4,can you help a man just snatched a bag and ran...,9,calm,scream,high
4,5,can you help a child fell into the swimming po...,3,fear,water_splash,critical


# convert in lower case

In [84]:
df['call_text'] = df['call_text'].apply( lambda x: x.lower())

# remove puncuation

In [85]:
import string
def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))
df['call_text'] = df['call_text'].apply(remove_punc)

# remove numbers

In [86]:
def remove_numbers(txt):
    new =''
    for i in txt :
        if not i.isdigit():
            new = new + i
    return new

df['call_text'] = df['call_text'].apply(remove_numbers)

# remove emojis

In [87]:
def remove_emojis(txt):
    new = ''
    for i in txt:
        if i.isascii():
            new += i
    return new
df['call_text'] = df['call_text'].apply(remove_emojis)

# remove stopwards

In [88]:
import nltk

In [89]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [90]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [91]:
stop_words = set(stopwords.words('english'))

In [92]:
df.loc[1]['call_text']

'listen flood water is entering houses and people are trapped please send help quickly'

In [93]:
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)



In [94]:
df['call_text'] = df['call_text'].apply(remove)

In [95]:
df.loc[1]['call_text']

'listen flood water entering houses people trapped please send help quickly'

In [96]:
y=df['emergency_type']
x=df['call_text']

In [97]:
from sklearn.model_selection import train_test_split


In [98]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

In [99]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000,stop_words='english')



x_train = vectorizer.fit_transform(x_train).toarray()
x_test = vectorizer.transform(x_test).toarray()


In [100]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([

    Dense(512, activation='relu', input_shape=(x_train.shape[1],)),
    Dropout(0.4),

    Dense(256, activation='relu'),
    Dropout(0.3),

    Dense(128, activation='relu'),  # feature layer for fusion

    Dense(13, activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [101]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [102]:
history = model.fit(
    x_train,
    y_train,
    validation_data=(x_test, y_test),
    epochs=5,
    batch_size=32
)

Epoch 1/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - accuracy: 0.1495 - loss: 2.5337 - val_accuracy: 0.6346 - val_loss: 2.3314
Epoch 2/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.5679 - loss: 2.2043 - val_accuracy: 0.9872 - val_loss: 1.3818
Epoch 3/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9784 - loss: 1.0398 - val_accuracy: 1.0000 - val_loss: 0.1366
Epoch 4/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9977 - loss: 0.1247 - val_accuracy: 1.0000 - val_loss: 0.0105
Epoch 5/5
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.0187 - val_accuracy: 1.0000 - val_loss: 0.0022


In [137]:
text=input()

text=vectorizer.transform([text])

A child fell into the swimming pool and not coming out Someone is drowning in the river please help Boat overturned people struggling in water Kid slipped into lake and disappeared Man jumped into canal and drowning Two swimmers stuck in strong current A person fell from bridge into river Child fell into water tank Tourist drowning near beach Man screaming for help in the lake


In [138]:
le.inverse_transform([np.argmax(model.predict(text))])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 90ms/step


array(['drowning'], dtype=object)

In [139]:
model.save('text_model.keras')

In [141]:
import pickle

pickle.dump(vectorizer, open("vectorizer_text.pkl","wb"))

In [142]:
pickle.dump(le, open("label_encoder_text.pkl","wb"))